# 03a · 학습 shard와 익명 test 준비

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 봉인된 분할을 tar shard로 만들고 test의 원래 ID와 정답을 추론 입력에서 분리합니다.
- **입력:** 02의 v3 split seal, canonical bbox와 원천 ZIP
- **출력:** train/validation shard, 익명 test shard와 inference seal; 비공개 key map과 HMAC 비밀키는 별도 prefix
- **실제 저장된 실행 상태:** 저장 출력은 DATASET COMPLETE, DATA_PROFILE=full, train=246,308 / val=30,808 / test=30,809입니다. dataset version 이름은 m3ds-v2-pilot이지만 실제 저장 profile은 full입니다. 비밀키는 코드에서 생성/다운로드하며 공개본에는 키나 key map 파일을 포함하지 않습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03a_dataset_preparation.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03a_dataset_preparation.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 03A. 공통 학습 shard와 익명 test shard 생성

이 노트북은 봉인된 `v3` split과 `bbox_quality_v2`를 모델 입력용 비압축 tar shard로 변환합니다.

- train: `eligible_for_training=True`만 사용
- validation: 후보가 모두 같은 30,808개를 사용
- test: 원래 stem과 ZIP 경로를 제거한 익명 `sample_id`로 저장
- 원래 test stem 연결표와 정답은 `eval_private`에만 저장
- 원본 ZIP byte는 수정하지 않음

`pilot`은 유형당 최대 3,000개의 train group을 사용하고 validation·test는 전부 준비합니다.
최종 승자 재학습 전에는 `DATA_PROFILE="full"`로 새 dataset version을 만들어야 합니다.


## 런타임

표준 CPU 런타임을 권장합니다. GPU는 사용하지 않습니다. 22개 원천 ZIP을 순차로 읽으므로
실행 중 Colab 연결을 끊지 마십시오. archive별 commit을 남기므로 같은 설정으로 다시 실행하면
이미 검증된 archive는 건너뜁니다.


In [ ]:
%pip -q install "google-cloud-storage>=2.19,<4" "pyarrow>=18,<24" "pandas>=2.2,<3"


In [ ]:
import gc
import hmac
import hashlib
import io
import json
import math
import os
import re
import secrets
import shutil
import tarfile
import tempfile
import time
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import auth
from IPython.display import display

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
SPLIT_VERSION = "v3"
AUDIT_VERSION = "v2"

# 먼저 pilot으로 동작을 검증하십시오. full은 train 246,308개 안팎을 모두 기록합니다.
DATA_PROFILE = "full"            # "pilot" 또는 "full"
DATASET_VERSION = "m3ds-v2-pilot" # 설정을 바꾸면 새 version을 사용
PILOT_PER_TYPE = 3_000
SHARD_MAX_BYTES = 768 * 1024**2
SHARD_MAX_SAMPLES = 2_048
BUILD_FULL_VALIDATION = True
BUILD_FULL_TEST = True
SEED = 42

TYPE_ORDER = list("ABCEFGHS")
EXPECTED_VAL_N = 30_808
EXPECTED_TEST_N = 30_809
LOCAL_ROOT = Path("/content/dcc_mission3_03a")
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_STAGE = LOCAL_ROOT / "stage"
LOCAL_META = LOCAL_ROOT / "meta"
for path in [LOCAL_INPUT, LOCAL_STAGE, LOCAL_META]:
    path.mkdir(parents=True, exist_ok=True)

PREFIX = f"dcc/mission3/datasets/{DATASET_VERSION}"
PRIVATE_PREFIX = f"dcc/eval_private/mission3/{DATASET_VERSION}"
INFERENCE_PREFIX = f"dcc/inference/mission3/{DATASET_VERSION}"

BUILD_CONFIG = {
    "schema_version": "dcc-m3-dataset-build-v1",
    "split_version": SPLIT_VERSION,
    "audit_version": AUDIT_VERSION,
    "dataset_version": DATASET_VERSION,
    "data_profile": DATA_PROFILE,
    "pilot_per_type": PILOT_PER_TYPE if DATA_PROFILE == "pilot" else None,
    "shard_max_bytes": SHARD_MAX_BYTES,
    "shard_max_samples": SHARD_MAX_SAMPLES,
    "build_full_validation": BUILD_FULL_VALIDATION,
    "build_full_test": BUILD_FULL_TEST,
    "zip_member_normalization": "mission1_flat_leading_slash_v1",
    "seed": SEED,
}
BUILD_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(BUILD_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
print(json.dumps(BUILD_CONFIG, ensure_ascii=False, indent=2))
print("build_config_sha256:", BUILD_CONFIG_SHA256)


{
  "schema_version": "dcc-m3-dataset-build-v1",
  "split_version": "v3",
  "audit_version": "v2",
  "dataset_version": "m3ds-v2-pilot",
  "data_profile": "full",
  "pilot_per_type": null,
  "shard_max_bytes": 805306368,
  "shard_max_samples": 2048,
  "build_full_validation": true,
  "build_full_test": true,
  "zip_member_normalization": "mission1_flat_leading_slash_v1",
  "seed": 42
}
build_config_sha256: 6edc570c412a2f05f97d531abf13f703d9b9ffae4a5f2394f535b027914e6d38


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        remote_artifact_type = (blob.metadata or {}).get("artifact_type")
        if (
            remote_sha != file_sha
            or remote_artifact_type != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (
        (blob.metadata or {}).get("sha256") != file_sha
        or (blob.metadata or {}).get("artifact_type") != artifact_type
    ):
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        ) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    remote_artifact_type = metadata.get("artifact_type")
    if artifact_type is not None and remote_artifact_type != artifact_type:
        raise RuntimeError(
            f"remote artifact_type 불일치: {object_name} "
            f"({remote_artifact_type!r} != {artifact_type!r})"
        )
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": remote_artifact_type or "unknown",
    }


In [ ]:
auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
try:
    BUCKET.reload()
except Exception as exc:
    raise RuntimeError(
        "버킷 접근 실패입니다. 업로드 때 사용한 Google 계정으로 다시 인증하고 "
        "Storage Object Admin과 bucket metadata 조회 권한을 확인하십시오."
    ) from exc
print("Bucket:", BUCKET.name, "location:", BUCKET.location)

# 익명 ID를 원본 stem에서 역산하지 못하도록 dataset version별 비밀키를 private prefix에 둡니다.
# 같은 DATASET_VERSION을 재실행하면 기존 key를 재사용하며 inference seal에는 key descriptor를 넣지 않습니다.
ANON_SECRET_OBJECT = f"{PRIVATE_PREFIX}/anonymization_secret.bin"
secret_blob = BUCKET.blob(ANON_SECRET_OBJECT)
try:
    ANON_SECRET = secret_blob.download_as_bytes()
except NotFound:
    candidate_secret = secrets.token_bytes(32)
    try:
        secret_blob.upload_from_string(
            candidate_secret,
            content_type="application/octet-stream",
            if_generation_match=0,
            checksum="auto",
        )
        ANON_SECRET = candidate_secret
    except PreconditionFailed:
        ANON_SECRET = secret_blob.download_as_bytes()
secret_blob.reload()
if len(ANON_SECRET) != 32:
    raise RuntimeError("익명화 비밀키 길이가 32 byte가 아닙니다")
ANON_SECRET_DESCRIPTOR = {
    "uri": f"gs://{BUCKET_NAME}/{ANON_SECRET_OBJECT}",
    "generation": int(secret_blob.generation),
    "bytes": int(secret_blob.size),
    "sha256": hashlib.sha256(ANON_SECRET).hexdigest(),
    "artifact_type": "m3_private_anonymization_secret",
}
print("Private anonymization key ready (key bytes are not printed).")


Bucket: YOUR_GCS_BUCKET location: US-CENTRAL1
Private anonymization key ready (key bytes are not printed).


## 1. 봉인된 v3 split과 canonical bbox 로드

test 이미지를 찾는 작업은 이 신뢰된 준비 런타임에서만 `splits_v3`를 사용합니다.
test row에서는 target과 target-derived 열을 Parquet projection에서 제외합니다. 이후 test 추론
노트북에는 이 private 표를 전달하지 않습니다.


In [ ]:
INPUT_OBJECTS = {
    "splits": f"dcc/eval_private/manifests/splits_{SPLIT_VERSION}.parquet",
    "split_seal": f"dcc/eval_private/manifests/split_seal_{SPLIT_VERSION}.json",
    "bbox": f"dcc/audits/bbox_quality_{AUDIT_VERSION}.parquet",
    "manifest_seal": f"dcc/audits/manifest_seal_{AUDIT_VERSION}.json",
}
local_paths = {}
for name, object_name in INPUT_OBJECTS.items():
    suffix = Path(object_name).suffix
    local = LOCAL_INPUT / f"{name}{suffix}"
    download_object(BUCKET, object_name, local)
    local_paths[name] = local
    print(name, local.stat().st_size, sha256_file(local))

split_seal = json.loads(local_paths["split_seal"].read_text(encoding="utf-8"))
manifest_seal = json.loads(local_paths["manifest_seal"].read_text(encoding="utf-8"))
if not all(split_seal.get("assertions", {}).values()):
    raise AssertionError("upstream split seal assertion 실패")

def verify_upstream_artifact(local_path, descriptor, expected_object):
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != BUCKET_NAME or object_name != expected_object:
        raise AssertionError("upstream seal artifact URI 불일치")
    expected_size = int(descriptor.get("bytes", descriptor.get("size", -1)))
    if Path(local_path).stat().st_size != expected_size:
        raise AssertionError("upstream artifact byte 불일치")
    if sha256_file(local_path) != descriptor["sha256"]:
        raise AssertionError("upstream artifact SHA 불일치")
    remote = BUCKET.blob(object_name, generation=int(descriptor["generation"]))
    remote.reload()
    if int(remote.size) != expected_size:
        raise AssertionError("upstream pinned generation byte 불일치")

verify_upstream_artifact(
    local_paths["splits"], split_seal["artifacts"]["splits"], INPUT_OBJECTS["splits"]
)
verify_upstream_artifact(
    local_paths["bbox"], manifest_seal["artifacts"]["bbox_quality"], INPUT_OBJECTS["bbox"]
)
BBOX_QUALITY_SHA256 = sha256_file(local_paths["bbox"])

# Archive resume key에는 사용자가 조절하는 설정 외에도 실제 익명화 key와 upstream
# generation/SHA를 포함합니다. 같은 DATASET_VERSION에 입력 byte가 달라지면 기존 commit을
# 재사용하지 않고 즉시 충돌로 중단합니다.
def compact_descriptor(descriptor):
    return {
        "uri": descriptor["uri"],
        "generation": int(descriptor["generation"]),
        "bytes": int(descriptor.get("bytes", descriptor.get("size", -1))),
        "sha256": descriptor["sha256"],
    }

def seal_object_descriptor(object_name, local_path, artifact_type):
    blob = BUCKET.blob(object_name)
    blob.reload()
    local_path = Path(local_path)
    if int(blob.size) != local_path.stat().st_size:
        raise AssertionError(f"seal object byte 불일치: {object_name}")
    local_sha = sha256_file(local_path)
    metadata_sha = (blob.metadata or {}).get("sha256")
    if metadata_sha and metadata_sha != local_sha:
        raise AssertionError(f"seal object SHA metadata 불일치: {object_name}")
    return {
        "uri": f"gs://{BUCKET_NAME}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": local_sha,
        "artifact_type": artifact_type,
    }

SPLIT_SEAL_DESCRIPTOR = seal_object_descriptor(
    INPUT_OBJECTS["split_seal"], local_paths["split_seal"], "m1_split_seal"
)
MANIFEST_SEAL_DESCRIPTOR = seal_object_descriptor(
    INPUT_OBJECTS["manifest_seal"], local_paths["manifest_seal"], "m1_manifest_seal"
)

EFFECTIVE_CONFIG = {
    "build_config": BUILD_CONFIG,
    "build_config_sha256": BUILD_CONFIG_SHA256,
    "anonymization_secret": compact_descriptor(ANON_SECRET_DESCRIPTOR),
    "upstream": {
        "split_seal": compact_descriptor(SPLIT_SEAL_DESCRIPTOR),
        "splits": compact_descriptor(split_seal["artifacts"]["splits"]),
        "manifest_seal": compact_descriptor(MANIFEST_SEAL_DESCRIPTOR),
        "bbox_quality": compact_descriptor(manifest_seal["artifacts"]["bbox_quality"]),
    },
}
EFFECTIVE_CONFIG_SHA256 = canonical_sha256(EFFECTIVE_CONFIG)
print("effective_config_sha256:", EFFECTIVE_CONFIG_SHA256)
required_split_columns = {
    "stem", "type", "waterlevel_m", "archive_key", "source_zip_uri",
    "source_generation", "source_member", "source_member_crc32",
    "source_member_bytes", "image_sha256", "group_id", "water_bin",
    "split", "eligible_for_training", "detector_trainable",
    "valid_mokja_count", "width", "height",
}
split_schema_columns = set(pq.read_schema(local_paths["splits"]).names)
missing = required_split_columns - split_schema_columns
if missing:
    raise AssertionError(f"splits_v3 필수 열 누락: {sorted(missing)}")

# 준비 runtime도 test truth를 DataFrame에 올리지 않습니다. train/val에만 target column을
# projection하고 test는 image 물질화에 필요한 위치·무결성 열만 읽습니다.
non_test = pq.read_table(
    local_paths["splits"],
    columns=sorted(required_split_columns),
    filters=[("split", "in", ["train", "val"])],
).to_pandas()
safe_test_columns = {
    "stem", "type", "archive_key", "source_zip_uri", "source_generation",
    "source_member", "source_member_crc32", "source_member_bytes",
    "image_sha256", "group_id", "split", "width", "height",
}
test_without_truth = pq.read_table(
    local_paths["splits"],
    columns=sorted(safe_test_columns),
    filters=[("split", "=", "test")],
).to_pandas()
for column in sorted(required_split_columns - safe_test_columns):
    test_without_truth[column] = (
        False if column in {"eligible_for_training", "detector_trainable"} else np.nan
    )
splits = pd.concat(
    [non_test, test_without_truth], ignore_index=True
)[sorted(required_split_columns)]
assert len(splits) == 308_085
assert splits["stem"].is_unique
assert set(splits["type"]) == set(TYPE_ORDER)
assert splits.loc[splits["split"].eq("test"), "waterlevel_m"].isna().all()
assert np.isfinite(
    splits.loc[~splits["split"].eq("test"), "waterlevel_m"].to_numpy(float)
).all()
assert splits.groupby("group_id")["split"].nunique().max() == 1
assert int((splits["split"] == "val").sum()) == EXPECTED_VAL_N
assert int((splits["split"] == "test").sum()) == EXPECTED_TEST_N
local_paths["splits"].unlink()
assert not local_paths["splits"].exists()
display(splits.groupby(["type", "split"], observed=True).size().unstack(fill_value=0))


splits 35944668 a1ee8301ec6addc824d5cf39c2ac1ad12ec28ff465cc94db41a8f3ff17d5c251
split_seal 9420 b10f0bfe0659504a1d3fb671e95005b254b838310ba32a1ed48b1bbbb91bd775
bbox 191995606 536fdbb9d8a8f553e6a4ff7c65e341ede621f996acef484cbf0e59ba35fa46af
manifest_seal 53609 f657b216d7fdc6c257d4544f209d19be0f1f597600cc6e7122693ab546d49e98
effective_config_sha256: 2e616bf1d16f4806536c95bba877471d3da9571d150e548fb6fdf8402bf67592


split   test  train    val
type                      
A      10522  84176  10522
B       3727  29813   3726
C       9194  73553   9194
E       2351  18805   2350
F        878   7030    879
G       1098   8781   1098
H       1031   8249   1031
S       2008  16061   2008

In [ ]:
train_val_stems = set(non_test["stem"].astype(str))
test_stems = set(test_without_truth["stem"].astype(str))
bbox_table = pq.read_table(
    local_paths["bbox"],
    columns=[
        "stem", "annotation_index", "annotation_name",
        "clipped_x1", "clipped_y1", "clipped_x2", "clipped_y2",
        "usable_after_clip",
    ],
    filters=[
        ("annotation_name", "=", "mokja"),
        ("usable_after_clip", "=", True),
        ("stem", "in", sorted(train_val_stems)),
    ],
).to_pandas()
assert not set(bbox_table["stem"].astype(str)).intersection(test_stems)
bbox_table = bbox_table.sort_values(
    ["stem", "annotation_index"], kind="mergesort"
)
coords = ["clipped_x1", "clipped_y1", "clipped_x2", "clipped_y2"]
assert np.isfinite(bbox_table[coords].to_numpy()).all()
assert (bbox_table["clipped_x1"] >= 0).all()
assert (bbox_table["clipped_y1"] >= 0).all()
assert (bbox_table["clipped_x2"] <= 1).all()
assert (bbox_table["clipped_y2"] <= 1).all()
assert (bbox_table["clipped_x2"] > bbox_table["clipped_x1"]).all()
assert (bbox_table["clipped_y2"] > bbox_table["clipped_y1"]).all()
boxes_by_stem = {
    stem: group[coords].astype(float).values.tolist()
    for stem, group in bbox_table.groupby("stem", sort=False)
}
observed_counts = pd.Series({key: len(value) for key, value in boxes_by_stem.items()})
check_rows = splits.loc[splits["valid_mokja_count"].gt(0), ["stem", "valid_mokja_count"]]
check_counts = check_rows["stem"].map(observed_counts).fillna(0).astype(int)
assert np.array_equal(check_counts.to_numpy(), check_rows["valid_mokja_count"].astype(int).to_numpy())
local_paths["bbox"].unlink()
assert not local_paths["bbox"].exists()
print("Canonical mokja boxes:", len(bbox_table), "rows /", len(boxes_by_stem), "images")


Canonical mokja boxes: 282621 rows / 277262 images


## 2. pilot 선택과 test 익명 키 생성

pilot은 기존 split을 바꾸지 않습니다. 같은 `group_id`는 항상 함께 선택하고, 각 유형 안에서
`water_bin` 비율을 근사 보존하도록 안정 해시 순서로 최대 3,000개 행을 고릅니다. validation은
공정 비교를 위해 전부 사용합니다.


In [ ]:
def stable_hex(*parts):
    text = "|".join(map(str, parts)).encode("utf-8")
    return hashlib.sha256(text).hexdigest()

train_all = splits.loc[
    splits["split"].eq("train") & splits["eligible_for_training"].astype(bool)
].copy()
val_all = splits.loc[splits["split"].eq("val")].copy()
test_all = splits.loc[splits["split"].eq("test")].copy()
assert train_all["eligible_for_training"].all()
assert len(val_all) == EXPECTED_VAL_N
assert len(test_all) == EXPECTED_TEST_N

def choose_type_waterbin_group_pilot(frame, per_type):
    selected_group_ids = set()
    for type_name in TYPE_ORDER:
        part = frame.loc[frame["type"].eq(type_name)]
        group_sizes = (
            part.groupby("group_id", sort=False)
            .agg(
                n=("group_id", "size"),
                water_bin=("water_bin", lambda values: sorted(map(str, values))[0]),
            )
            .reset_index()
        )
        group_sizes["order"] = group_sizes["group_id"].map(
            lambda value: stable_hex(SEED, "pilot", type_name, value)
        )
        bin_rows = part["water_bin"].astype(str).value_counts().sort_index()
        exact = bin_rows / bin_rows.sum() * per_type
        quotas = np.floor(exact).astype(int)
        for bin_name in (exact - quotas).sort_values(ascending=False).index[
            : int(per_type - quotas.sum())
        ]:
            quotas.loc[bin_name] += 1
        for bin_name, quota in quotas.items():
            running = 0
            candidates = group_sizes.loc[
                group_sizes["water_bin"].eq(str(bin_name))
            ].sort_values("order", kind="mergesort")
            for row in candidates.itertuples():
                if running > 0 and running + int(row.n) > int(quota):
                    continue
                selected_group_ids.add(row.group_id)
                running += int(row.n)
                if running >= int(quota):
                    break
    chosen = frame.loc[frame["group_id"].isin(selected_group_ids)].copy()
    assert chosen.groupby("group_id")["split"].nunique().max() == 1
    return chosen

if DATA_PROFILE == "pilot":
    train_selected = choose_type_waterbin_group_pilot(train_all, PILOT_PER_TYPE)
elif DATA_PROFILE == "full":
    train_selected = train_all.copy()
else:
    raise ValueError("DATA_PROFILE은 pilot 또는 full이어야 합니다.")

val_selected = val_all.copy() if BUILD_FULL_VALIDATION else choose_type_waterbin_group_pilot(val_all, 800)
if not BUILD_FULL_TEST:
    raise RuntimeError("익명 test 계약은 전체 test만 허용합니다")
test_selected = test_all.copy()

def neutral_test_id(stem):
    type_name = str(stem)[0]
    if type_name not in TYPE_ORDER:
        raise ValueError(f"지원하지 않는 test 유형: {type_name}")
    digest = hmac.new(
        ANON_SECRET, f"sample|{stem}".encode("utf-8"), hashlib.sha256
    ).hexdigest()
    return type_name + digest[:24]

def opaque_test_shard_token(archive_key, shard_index):
    return hmac.new(
        ANON_SECRET,
        f"test-shard|{archive_key}|{int(shard_index)}".encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()[:24]

test_selected["sample_id"] = test_selected["stem"].map(neutral_test_id)
assert test_selected["sample_id"].is_unique
assert test_selected["sample_id"].str.fullmatch(r"[ABCEFGHS][0-9a-f]{24}").all()
assert test_selected["sample_id"].str[0].eq(test_selected["type"]).all()
test_key_map = test_selected[["sample_id", "stem"]].sort_values("sample_id")
assert test_key_map["stem"].is_unique

selected = pd.concat(
    [train_selected.assign(output_split="train"),
     val_selected.assign(output_split="val"),
     test_selected.assign(output_split="test")],
    ignore_index=True,
)
assert selected["stem"].is_unique
assert selected.groupby("group_id")["split"].nunique().max() == 1
display(selected.groupby(["type", "output_split"], observed=True).size().unstack(fill_value=0))
print("selected:", selected["output_split"].value_counts().to_dict())


output_split   test  train    val
type                             
A             10522  84174  10522
B              3727  29813   3726
C              9194  73398   9194
E              2351  18803   2350
F               878   7030    879
G              1098   8781   1098
H              1031   8248   1031
S              2008  16061   2008

selected: {'train': 246308, 'test': 30809, 'val': 30808}


## 3. archive별 shard 생성·검증·게시

원천 ZIP 하나만 로컬에 내려받고, 선택된 member의 byte 수·CRC32·SHA-256을 다시 검증합니다.
tar는 JPEG byte를 다시 인코딩하지 않습니다. test metadata에는 `sample_id`만 들어가며, 목자판
유형은 `sample_id`의 첫 문자에서만 얻습니다.


In [ ]:
SAFE_TRAIN_KEY = re.compile(r"^[A-Z][A-Za-z0-9_.-]+$")
SAFE_TEST_KEY = re.compile(r"^[ABCEFGHS][0-9a-f]{24}$")

class TarShardWriter:
    def __init__(self, split_name, archive_key, on_close):
        self.split_name = split_name
        self.archive_key = archive_key
        self.on_close = on_close
        self.index = 0
        self.tar = None
        self.path = None
        self.sample_count = 0
        self.payload_bytes = 0
        self.descriptors = []

    def _open(self):
        split_dir = LOCAL_STAGE / self.archive_key / self.split_name
        split_dir.mkdir(parents=True, exist_ok=True)
        self.path = split_dir / f"{self.split_name}-{self.archive_key}-{self.index:05d}.tar.partial"
        self.tar = tarfile.open(self.path, mode="w")
        self.sample_count = 0
        self.payload_bytes = 0

    @staticmethod
    def _add_bytes(handle, name, payload):
        info = tarfile.TarInfo(name=name)
        info.size = len(payload)
        info.mtime = 0
        info.mode = 0o644
        handle.addfile(info, io.BytesIO(payload))

    def write(self, key, image_bytes, metadata):
        meta_bytes = json.dumps(
            metadata, ensure_ascii=False, sort_keys=True, separators=(",", ":")
        ).encode("utf-8")
        incoming = len(image_bytes) + len(meta_bytes)
        if self.tar is None:
            self._open()
        elif (
            self.sample_count >= SHARD_MAX_SAMPLES
            or self.payload_bytes + incoming > SHARD_MAX_BYTES
        ):
            self.close_shard()
            self._open()
        self._add_bytes(self.tar, f"{key}.jpg", image_bytes)
        self._add_bytes(self.tar, f"{key}.json", meta_bytes)
        self.sample_count += 1
        self.payload_bytes += incoming

    def close_shard(self):
        if self.tar is None:
            return
        self.tar.close()
        final_path = self.path.with_suffix("")
        self.path.replace(final_path)
        expected_members = self.sample_count * 2
        with tarfile.open(final_path, "r:") as check:
            members = check.getmembers()
            if len(members) != expected_members:
                raise AssertionError("tar member 수 검증 실패")
            if any(member.isdir() or not member.isfile() for member in members):
                raise AssertionError("tar에 일반 파일 외 member가 있습니다")
        descriptor = self.on_close(
            final_path, self.split_name, self.archive_key,
            self.index, self.sample_count, self.payload_bytes,
        )
        self.descriptors.append(descriptor)
        final_path.unlink(missing_ok=True)
        self.index += 1
        self.tar = None
        self.path = None

    def close(self):
        self.close_shard()
        return self.descriptors

def shard_object_name(split_name, archive_key, shard_index):
    if split_name == "test":
        token = opaque_test_shard_token(archive_key, shard_index)
        return f"{INFERENCE_PREFIX}/test_images/test-{token}.tar"
    return f"{PREFIX}/{DATA_PROFILE}/{split_name}/{archive_key}/{split_name}-{shard_index:05d}.tar"

def publish_shard(path, split_name, archive_key, shard_index, samples, payload_bytes):
    object_name = shard_object_name(split_name, archive_key, shard_index)
    descriptor = upload_create_only(BUCKET, path, object_name, f"m3_{split_name}_image_shard")
    descriptor.update({
        "split": split_name,
        "samples": int(samples),
        "payload_bytes": int(payload_bytes),
    })
    if split_name != "test":
        descriptor["archive_key"] = archive_key
    print("published", descriptor["uri"], "samples=", samples)
    return descriptor

def crc_text(value):
    if isinstance(value, (int, np.integer)):
        return f"{int(value) & 0xffffffff:08x}"
    text = str(value).strip().lower()
    return text.removeprefix("0x").zfill(8)

# Mission 1 감사에서는 원본 ZIP의 단일 leading slash를 제거한 flat member 이름을
# manifest에 기록했습니다. 동일한 정규화·경로 안전 규칙으로 raw ZipInfo를 다시 매핑합니다.
def normalize_flat_member(raw_name):
    if not isinstance(raw_name, str):
        raise ValueError("ZIP member 이름이 문자열이 아닙니다")
    if "\x00" in raw_name or "\\" in raw_name or raw_name.startswith("//"):
        raise ValueError(f"안전하지 않은 ZIP member: {raw_name!r}")
    name = raw_name[1:] if raw_name.startswith("/") else raw_name
    if (
        not name or "/" in name or name in {".", ".."}
        or ":" in name or Path(name).name != name
    ):
        raise ValueError(f"중첩 또는 안전하지 않은 ZIP member: {raw_name!r}")
    return name

assert normalize_flat_member("A_0005_x.jpg") == "A_0005_x.jpg"
assert normalize_flat_member("/A_0005_x.jpg") == "A_0005_x.jpg"
for unsafe_name in ["//A.jpg", "../A.jpg", "a/b.jpg", "a\\b.jpg", "C:A.jpg", "\x00A.jpg"]:
    try:
        normalize_flat_member(unsafe_name)
    except ValueError:
        pass
    else:
        raise AssertionError(f"거부되지 않은 위험 ZIP 경로: {unsafe_name!r}")

def archive_commit_name(archive_key):
    return f"{PREFIX}/commits/{archive_key}.json"

def verified_existing_commit(archive_key):
    blob = BUCKET.blob(archive_commit_name(archive_key))
    if not blob.exists():
        return None
    payload = json.loads(blob.download_as_bytes().decode("utf-8"))
    if payload.get("effective_config_sha256") != EFFECTIVE_CONFIG_SHA256:
        raise RuntimeError(
            f"{archive_key} commit 설정 충돌. DATASET_VERSION을 바꾸십시오."
        )
    for descriptor in payload["shards"]:
        bucket_name, object_name = parse_gs_uri(descriptor["uri"])
        assert bucket_name == BUCKET_NAME
        remote = BUCKET.blob(object_name, generation=descriptor["generation"])
        remote.reload()
        if int(remote.size) != int(descriptor["bytes"]):
            raise RuntimeError(f"기존 shard size 불일치: {descriptor['uri']}")
        if (remote.metadata or {}).get("sha256") != descriptor["sha256"]:
            raise RuntimeError(f"기존 shard SHA metadata 불일치: {descriptor['uri']}")
    return payload


In [ ]:
archive_commits = []
grouped_archives = list(selected.groupby("archive_key", sort=True))
for archive_number, (archive_key, archive_rows) in enumerate(grouped_archives, start=1):
    existing = verified_existing_commit(archive_key)
    if existing is not None:
        archive_commits.append(existing)
        print(f"[{archive_number}/{len(grouped_archives)}] {archive_key}: verified commit -> skip")
        continue

    print(f"[{archive_number}/{len(grouped_archives)}] {archive_key}: build")
    source_specs = archive_rows[
        ["source_zip_uri", "source_generation"]
    ].drop_duplicates()
    if len(source_specs) != 1:
        raise AssertionError(f"{archive_key}: source ZIP spec가 하나가 아닙니다")
    source_uri = source_specs.iloc[0]["source_zip_uri"]
    source_generation = int(source_specs.iloc[0]["source_generation"])
    source_bucket_name, source_object_name = parse_gs_uri(source_uri)
    if source_bucket_name != BUCKET_NAME:
        raise AssertionError("예상하지 않은 source bucket")

    source_blob = BUCKET.blob(source_object_name, generation=source_generation)
    source_blob.reload()
    required_free = int(source_blob.size) + 2 * SHARD_MAX_BYTES + 8 * 1024**3
    free = shutil.disk_usage("/content").free
    if free < required_free:
        raise RuntimeError(
            f"디스크 부족: free={free/1024**3:.1f}GiB, "
            f"required={required_free/1024**3:.1f}GiB"
    )
    local_zip = LOCAL_STAGE / f"{archive_key}.zip"
    local_zip.unlink(missing_ok=True)
    source_blob.download_to_filename(
        str(local_zip), if_generation_match=source_generation, checksum="auto"
    )

    writers = {
        split_name: TarShardWriter(split_name, archive_key, publish_shard)
        for split_name in sorted(archive_rows["output_split"].unique())
    }
    observed_stems = []
    with zipfile.ZipFile(local_zip) as zf:
        info_by_name = {}
        for info in zf.infolist():
            if info.is_dir():
                continue
            normalized_name = normalize_flat_member(info.filename)
            if normalized_name in info_by_name:
                raise AssertionError(
                    f"정규화 뒤 중복 ZIP member: {archive_key} / {normalized_name}"
                )
            info_by_name[normalized_name] = info
        for row in archive_rows.sort_values("stem", kind="mergesort").itertuples():
            if normalize_flat_member(row.source_member) != row.source_member:
                raise AssertionError(
                    f"manifest source_member가 canonical flat 이름이 아닙니다: {row.source_member}"
                )
            if row.source_member not in info_by_name:
                raise KeyError(f"ZIP member 누락: {row.source_member}")
            info = info_by_name[row.source_member]
            if int(info.file_size) != int(row.source_member_bytes):
                raise AssertionError(f"member byte 불일치: {row.stem}")
            if crc_text(info.CRC) != crc_text(row.source_member_crc32):
                raise AssertionError(f"member CRC 불일치: {row.stem}")
            raw = zf.read(info)
            if hashlib.sha256(raw).hexdigest() != str(row.image_sha256):
                raise AssertionError(f"image SHA 불일치: {row.stem}")

            split_name = row.output_split
            if split_name == "test":
                key = neutral_test_id(row.stem)
                if not SAFE_TEST_KEY.fullmatch(key):
                    raise AssertionError("익명 test key 형식 오류")
                if key[0] != row.type:
                    raise AssertionError("test ID 첫 문자와 유형 불일치")
                metadata = {"sample_id": key}
            else:
                key = row.stem
                if not SAFE_TRAIN_KEY.fullmatch(key):
                    raise AssertionError(f"train/val key 형식 오류: {key}")
                boxes = boxes_by_stem.get(row.stem, [])
                metadata = {
                    "stem": row.stem,
                    "type": row.type,
                    "target_m": float(row.waterlevel_m),
                    "split": split_name,
                    "group_id": str(row.group_id),
                    "image_sha256": str(row.image_sha256),
                    "width": int(row.width),
                    "height": int(row.height),
                    "detector_trainable": bool(row.detector_trainable),
                    "mokja_boxes_xyxy_norm": boxes,
                }
            writers[split_name].write(key, raw, metadata)
            observed_stems.append(row.stem)

    shard_descriptors = []
    for writer in writers.values():
        shard_descriptors.extend(writer.close())
    local_zip.unlink(missing_ok=True)
    expected_stems = sorted(archive_rows["stem"].astype(str))
    if sorted(observed_stems) != expected_stems:
        raise AssertionError(f"{archive_key}: 선택 stem 완전성 실패")

    commit_payload = {
        "status": "COMMITTED",
        "schema_version": "dcc-m3-archive-commit-v1",
        "dataset_version": DATASET_VERSION,
        "effective_config_sha256": EFFECTIVE_CONFIG_SHA256,
        "archive_key": archive_key,
        "source_uri": source_uri,
        "source_generation": source_generation,
        "counts": archive_rows["output_split"].value_counts().astype(int).to_dict(),
        "sample_key_sha256": canonical_sha256(expected_stems),
        "shards": shard_descriptors,
    }
    commit_descriptor = upload_json_create_only(
        BUCKET, commit_payload, archive_commit_name(archive_key), "m3_archive_commit"
    )
    commit_payload["commit_descriptor"] = commit_descriptor
    archive_commits.append(commit_payload)
    shutil.rmtree(LOCAL_STAGE / archive_key, ignore_errors=True)
    gc.collect()
print("archive commits:", len(archive_commits))


[1/22] A_0005: build
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00000.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00001.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00002.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00003.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00004.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00005.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/train/A_0005/train-00006.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/full/val/A_0005/val-00000.tar samples= 2048
published gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pi

## 4. private key map과 최종 seal

test key map과 HMAC 비밀키는 private prefix에 게시합니다. inference seal에는 익명 shard와
`sample_id` 외의 정보를 넣지 않습니다. 같은 `DATASET_VERSION`에 다른 설정을 덮어쓰지 않습니다.


In [ ]:
test_key_path = LOCAL_META / "test_key_map.parquet"
test_key_map.to_parquet(test_key_path, index=False, compression="zstd")
key_map_descriptor = upload_create_only(
    BUCKET,
    test_key_path,
    f"{PRIVATE_PREFIX}/test_key_map.parquet",
    "m3_private_test_key_map",
)

selection_path = LOCAL_META / "selected_rows.parquet"
selected_public_audit = selected[[
    "stem", "type", "group_id", "split", "output_split", "archive_key"
]].copy()
selected_public_audit.to_parquet(selection_path, index=False, compression="zstd")
selection_descriptor = upload_create_only(
    BUCKET,
    selection_path,
    f"{PRIVATE_PREFIX}/selected_rows.parquet",
    "m3_private_dataset_selection",
)

all_shards = [
    descriptor
    for commit in archive_commits
    for descriptor in commit["shards"]
]
split_shards = {
    split_name: sorted(
        [d for d in all_shards if d["split"] == split_name],
        key=lambda d: d["uri"],
    )
    for split_name in ["train", "val", "test"]
}
expected_counts = {
    "train": len(train_selected),
    "val": len(val_selected),
    "test": len(test_selected),
}
for split_name, descriptors in split_shards.items():
    assert sum(int(d["samples"]) for d in descriptors) == expected_counts[split_name]

split_group_check = selected.groupby("group_id")["split"].nunique().max()
assertions = {
    "archive_commits_complete": len(archive_commits) == splits["archive_key"].nunique(),
    "selected_stems_unique": bool(selected["stem"].is_unique),
    "group_split_leakage_zero": int(split_group_check) == 1,
    "train_eligible_only": bool(train_selected["eligible_for_training"].all()),
    "validation_full": len(val_selected) == EXPECTED_VAL_N,
    "test_full": len(test_selected) == EXPECTED_TEST_N,
    "local_split_with_truth_deleted": not local_paths["splits"].exists(),
    "local_bbox_with_test_annotations_deleted": not local_paths["bbox"].exists(),
    "test_keys_unique": bool(test_key_map["sample_id"].is_unique),
    "test_key_map_private": "/eval_private/" in key_map_descriptor["uri"],
    "test_shards_inference_only": all(
        "/inference/" in item["uri"] for item in split_shards["test"]
    ),
}
if not all(assertions.values()):
    raise AssertionError(f"dataset hard gate 실패: {assertions}")

dataset_seal = {
    "status": "DATASET_COMPLETE",
    "schema_version": "dcc-m3-dataset-seal-v1",
    "dataset_version": DATASET_VERSION,
    "data_profile": DATA_PROFILE,
    "config": BUILD_CONFIG,
    "build_config_sha256": BUILD_CONFIG_SHA256,
    "effective_config": EFFECTIVE_CONFIG,
    "config_sha256": EFFECTIVE_CONFIG_SHA256,
    "upstream": {
        "split_seal_sha256": sha256_file(local_paths["split_seal"]),
        "split_seal": SPLIT_SEAL_DESCRIPTOR,
        "manifest_seal_sha256": sha256_file(local_paths["manifest_seal"]),
        "manifest_seal": MANIFEST_SEAL_DESCRIPTOR,
        "bbox_quality_sha256": BBOX_QUALITY_SHA256,
    },
    "counts": expected_counts,
    "validation_stem_sha256": canonical_sha256(
        sorted(val_selected["stem"].astype(str).tolist())
    ),
    "validation_target_sha256": canonical_target_sha256(
        val_selected, "waterlevel_m"
    ),
    "test_sample_id_sha256": canonical_sha256(
        sorted(test_key_map["sample_id"].astype(str).tolist())
    ),
    "shards": {"train": split_shards["train"], "val": split_shards["val"]},
    "private": {
        "test_key_map": key_map_descriptor,
        "selected_rows": selection_descriptor,
        "anonymization_secret": ANON_SECRET_DESCRIPTOR,
    },
    "assertions": assertions,
}
dataset_seal_descriptor = upload_json_create_only(
    BUCKET, dataset_seal, f"{PREFIX}/dataset_seal.json", "m3_dataset_seal"
)

inference_seal = {
    "status": "INFERENCE_IMAGES_COMPLETE",
    "schema_version": "dcc-m3-inference-image-seal-v1",
    "dataset_version": DATASET_VERSION,
    "split_version": SPLIT_VERSION,
    "split_seal_sha256": dataset_seal["upstream"]["split_seal_sha256"],
    "counts": {"test": len(test_selected)},
    "test_sample_id_sha256": dataset_seal["test_sample_id_sha256"],
    "test_shards": [
        {
            key: item[key]
            for key in ["uri", "generation", "bytes", "sha256", "artifact_type"]
        }
        for item in split_shards["test"]
    ],
    "allowed_metadata": ["sample_id"],
    "forbidden_metadata": [
        "stem", "type", "waterlevel_m", "target_m", "clslevel", "archive_key",
        "source_zip_uri", "group_id", "bbox", "obstruction",
    ],
    "dataset_seal": dataset_seal_descriptor,
    "dataset_seal_sha256": dataset_seal_descriptor["sha256"],
    "assertions": {
        "test_count_exact": len(test_selected) == EXPECTED_TEST_N,
        "all_shards_in_inference_prefix": all(
            "/inference/" in item["uri"] for item in split_shards["test"]
        ),
        "private_map_not_embedded": True,
        "archive_key_not_embedded": all(
            "archive_key" not in item for item in split_shards["test"]
        ),
        "type_available_only_as_first_character": bool(
            test_key_map["sample_id"].str[0].isin(TYPE_ORDER).all()
        ),
    },
}
inference_seal_descriptor = upload_json_create_only(
    BUCKET,
    inference_seal,
    f"{INFERENCE_PREFIX}/test_image_seal.json",
    "m3_inference_image_seal",
)
print("DATASET COMPLETE")
print("dataset seal:", dataset_seal_descriptor["uri"])
print("inference seal:", inference_seal_descriptor["uri"])
print("private test key map:", key_map_descriptor["uri"])


DATASET COMPLETE
dataset seal: gs://YOUR_GCS_BUCKET/dcc/mission3/datasets/m3ds-v2-pilot/dataset_seal.json
inference seal: gs://YOUR_GCS_BUCKET/dcc/inference/mission3/m3ds-v2-pilot/test_image_seal.json
private test key map: gs://YOUR_GCS_BUCKET/dcc/eval_private/mission3/m3ds-v2-pilot/test_key_map.parquet


## 완료 기준

마지막 출력에 `DATASET COMPLETE`가 있어야 합니다. pilot 실험은 다음 노트북으로 이어집니다.

`DCC_Colab_03B_Mission3_세가지_FullFrame_회귀.ipynb`


In [ ]:
print("DATA_PROFILE:", DATA_PROFILE)
print("seal profile:", dataset_seal["data_profile"])
print("counts:", dataset_seal["counts"])

DATA_PROFILE: full
seal profile: full
counts: {'train': 246308, 'val': 30808, 'test': 30809}
